## Export trial info to csv

Trial data is loaded from the `.csv` files (one file per mouse). In addition to the combined trial table, the notebook exports 250 reproducibly sampled 1-second baseline intervals per mouse. Each interval ends 50 ms before `trial_start` and is retained only when it does not overlap the preceding trial.

In [1]:
import pandas as pd
from pathlib import Path

#### Parameters

In [2]:
project_folder = Path("E:/con2phys_submission")
source_folder = project_folder / "dataset_download" / "py-files"
output_folder = project_folder / "dataset"
trials_csv_path = output_folder / "trials_info.csv"
baseline_csv_path = output_folder / "baseline_periods.csv"

EXPECTED_N_MICE = 18
BASELINE_DURATION_SEC = 1.0
PRE_TRIAL_GUARD_SEC = 0.05
N_BASELINE_INTERVALS_PER_MOUSE = 250
RANDOM_SEED = 42

#### Loading trials data 

In [3]:
# collect all trial_data.csv into one DataFrame
all_trials = []
for ii in range(1, EXPECTED_N_MICE + 1):
    trialdata_filename = source_folder / str(ii) / "trial_data.csv"
    df_mouse = pd.read_csv(trialdata_filename).drop(columns="Unnamed: 0", errors="ignore")
    df_mouse.insert(0, "trial_id", range(1, len(df_mouse) + 1))
    df_mouse["trial_duration"] = df_mouse["trial_end"] - df_mouse["trial_start"]
    df_mouse.insert(0, "mouse_id", ii)   # put mouse_id as the first column
    all_trials.append(df_mouse)

output_folder.mkdir(parents=True, exist_ok=True)
trials_df = pd.concat(all_trials, ignore_index=True)
trials_df.to_csv(trials_csv_path, index=False)

trials_df

,mouse_id,trial_id,variable_A,variable_B,variable_C,trial_start,stim_start,outcome,trial_end,trial_duration
0,1,1,1,1,2,58.738992,69.705862,69.962403,71.072492,12.333500
1,1,2,1,1,2,72.098523,74.872945,75.279471,76.388860,4.290338
2,1,3,1,1,2,77.404392,79.222444,79.406092,80.505349,3.100957
3,1,4,0,0,2,81.501916,83.989167,84.185347,86.287937,4.786021
4,1,5,1,1,2,87.272272,88.888844,89.161016,90.271139,2.998867
...,...,...,...,...,...,...,...,...,...,...
10368,18,619,0,1,0,2715.423097,2716.084831,2720.247848,2722.301243,6.878146
10369,18,620,0,1,0,2723.725034,2724.191687,2731.254981,2733.301043,9.576009
10370,18,621,0,1,0,2734.733566,2735.207785,2735.684571,2737.733933,3.000367
10371,18,622,0,1,0,2739.147558,2739.891417,2740.525853,2742.584147,3.436589


#### Select equal-duration pre-trial baseline intervals

A candidate baseline is the 1-second interval `[trial_start - 1.05, trial_start - 0.05)`. Candidates overlapping the preceding trial, or following a trial with a missing `trial_end` are excluded. From the remaining candidates, 250 intervals are sampled without replacement for each mouse using a fixed random seed.

In [4]:
candidate_baselines = trials_df.sort_values(
    ["mouse_id", "trial_start"]
).reset_index(drop=True)
candidate_baselines["previous_trial_end"] = (
    candidate_baselines.groupby("mouse_id")["trial_end"].shift()
)
candidate_baselines["baseline_start"] = (
    candidate_baselines["trial_start"]
    - PRE_TRIAL_GUARD_SEC
    - BASELINE_DURATION_SEC
)
candidate_baselines["baseline_end"] = (
    candidate_baselines["trial_start"] - PRE_TRIAL_GUARD_SEC
)

is_first_trial = candidate_baselines.groupby("mouse_id").cumcount().eq(0)
does_not_overlap_previous_trial = (
    is_first_trial
    | (
        candidate_baselines["previous_trial_end"].notna()
        & (
            candidate_baselines["previous_trial_end"]
            <= candidate_baselines["baseline_start"]
        )
    )
)
eligible_baselines = candidate_baselines.loc[
    candidate_baselines["baseline_start"].ge(0)
    & does_not_overlap_previous_trial
].copy()

available_counts = eligible_baselines.groupby("mouse_id").size()
if len(available_counts) != EXPECTED_N_MICE:
    raise ValueError("At least one mouse has no eligible baseline intervals")
if (available_counts < N_BASELINE_INTERVALS_PER_MOUSE).any():
    insufficient = available_counts[
        available_counts < N_BASELINE_INTERVALS_PER_MOUSE
    ].to_dict()
    raise ValueError(
        f"Not enough eligible baseline intervals for some mice: {insufficient}"
    )

baseline_df = (
    eligible_baselines.groupby("mouse_id", group_keys=False)
    .sample(n=N_BASELINE_INTERVALS_PER_MOUSE, random_state=RANDOM_SEED)
    .sort_values(["mouse_id", "baseline_start"])
    .reset_index(drop=True)
)
baseline_df = baseline_df[[
    "mouse_id", "trial_id", "baseline_start", "baseline_end"
]]

expected_rows = EXPECTED_N_MICE * N_BASELINE_INTERVALS_PER_MOUSE
if len(baseline_df) != expected_rows:
    raise ValueError(
        f"Expected {expected_rows} selected baseline intervals, "
        f"found {len(baseline_df)}"
    )
selected_counts = baseline_df.groupby("mouse_id").size()
if not selected_counts.eq(N_BASELINE_INTERVALS_PER_MOUSE).all():
    raise ValueError("Each mouse must have exactly 250 selected intervals")
if baseline_df[["baseline_start", "baseline_end"]].isna().any().any():
    raise ValueError("Baseline boundaries contain missing values")
observed_durations = baseline_df["baseline_end"] - baseline_df["baseline_start"]
if not observed_durations.sub(BASELINE_DURATION_SEC).abs().le(1e-9).all():
    raise ValueError("Every selected baseline interval must be exactly 1 second")
if baseline_df[["mouse_id", "trial_id"]].duplicated().any():
    raise ValueError("A trial was selected more than once")

baseline_df.to_csv(baseline_csv_path, index=False)
display(available_counts.rename("n_available").reset_index())
baseline_df

,mouse_id,n_available
0,1,344
1,2,511
2,3,384
3,4,349
4,5,410
5,6,380
6,7,256
7,8,663
8,9,309
9,10,435


,mouse_id,trial_id,baseline_start,baseline_end
0,1,1,57.688992,58.688992
1,1,20,141.863141,142.863141
2,1,27,169.480479,170.480479
3,1,35,201.582901,202.582901
4,1,37,209.967196,210.967196
...,...,...,...,...
4495,18,617,2705.158585,2706.158585
4496,18,619,2714.373097,2715.373097
4497,18,621,2733.683566,2734.683566
4498,18,622,2738.097558,2739.097558
